# MIND-style Hallucination Detection — Refactored Pipeline

Thin driver notebook for the refactored codebase in `src/`.

**Model:** `meta-llama/Llama-3.2-1B` (4-bit NF4 quantization)

**What this notebook does**
1. Pulls the latest code from GitHub (Colab / Kaggle).
2. Installs dependencies.
3. Loads the LLM + spaCy.
4. Runs a sanity check on a single text.
5. Generates the dataset.
6. Trains the MIND classifier.
7. Evaluates and prints metrics.

If you change `src/*.py` locally, push to GitHub, then `!git pull` the first cell — no need to re-upload the notebook.

## 0. Repo + dependencies

In [ ]:
# --- Colab / Kaggle bootstrap ---
# Uncomment the clone line on a fresh runtime. Skip if you uploaded
# the repo manually.
# !git clone https://github.com/<your-user>/dissertation.git
# %cd dissertation/Code

!pip install -q -r requirements.txt
!python -m spacy download en_core_web_sm -q


## 1. Imports and config

In [ ]:
import sys, os
# Make sure the local src/ package is importable
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

from src import config
from src.model_loader import load_llm, load_nlp, set_seed
from src.entities import get_entities, filter_valid_entities
from src.tokens import find_first_and_next_token
from src.embeddings import get_hd
from src.dataset_gen import build_dataset, split_and_save, generate_sample
from src.classifier import MINDClassifier, MINDDataset
from src.train import load_data, train
from src.evaluate import evaluate, print_report

set_seed(config.SEED)
print('Model:', config.MODEL_NAME)
print('Target samples per class:', config.TARGET_SAMPLES)
print('Embedding key:', config.EMBEDDING_KEY)


## 2. Load LLM + NLP

In [ ]:
tokenizer, model = load_llm()
nlp = load_nlp()


## 3. Sanity check on a single text
Mirrors Block 5 of the original notebook — confirms that the model loads correctly, the entity-substitution probe finds a candidate, and the three embedding views all come back with the expected shape.

In [ ]:
import torch

sample_text = (
    "Albert Einstein was born in Ulm, in the Kingdom of "
    "Württemberg in the German Empire, on 14 March 1879. "
    "His parents were Hermann Einstein, a salesman and engineer, "
    "and Pauline Koch."
)

entities = get_entities(sample_text, nlp)
valid = filter_valid_entities(entities, title='Albert Einstein')
print(f'{len(valid)} valid entities')

if valid:
    e, idx = valid[0]
    sample = generate_sample(sample_text, e, idx, 'Albert Einstein', tokenizer, model)
    if sample is None:
        print('Sanity sample rejected (model already knew the entity, or filters tripped).')
    else:
        print('Original   :', sample['text_orig'][:120], '...')
        print('Hallucinated:', sample['text_hall'][:120], '...')
        print('Hallucinated entity:', sample['entity_hall'])
        print('Embedding dim (mean2):', len(sample['mean2_hall']))


## 4. Generate the dataset
Streams Wikipedia until `TARGET_SAMPLES` per class are collected. 1–2 hours on a free GPU.

In [ ]:
dataset_hall, dataset_non_hall = build_dataset(
    tokenizer, model, nlp,
    target_samples=config.TARGET_SAMPLES,
)
print(f'Hallucinated: {len(dataset_hall)} | Non-hallucinated: {len(dataset_non_hall)}')


## 5. Train / test split

In [ ]:
train_data, test_data = split_and_save(dataset_hall, dataset_non_hall)
print(f'Train: {len(train_data)} | Test: {len(test_data)}')

# Free the LLM before training the small MLP
import gc
del model, tokenizer
gc.collect()
import torch
torch.cuda.empty_cache()


## 6. Train the MIND classifier
5-layer MLP from `src/classifier.py`. Saves the best checkpoint by test accuracy to `llama32_mind_best.pth`.

In [ ]:
# (Re)load the JSONs to guarantee the trainer reads exactly what was saved.
train_data, test_data = load_data()
model_mlp, best_acc = train(train_data, test_data)
print(f'Best test accuracy: {best_acc:.4f}')


## 7. Full evaluation

In [ ]:
metrics = evaluate(model_mlp, test_data)
print_report(metrics, n_total=len(test_data))
